# 04 BM25
### David Alejandro Puga Novoa
#### Recuperación de información
#### 13 de Noviembre de 2026

### 1. Carga el corpus de Project Gutenberg

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [3]:
import os
import re
import math
import numpy as np
import pandas as pd
from collections import Counter

def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-záéíóúüñ]+(?:'[a-z]+)*", text)

titles = {
    "austen-emma.txt": "Emma",
    "austen-persuasion.txt": "Persuasion",
    "austen-sense.txt": "Sense and Sensibility",
    "bible-kjv.txt": "The King James Bible",
    "blake-poems.txt": "Poems (Blake)",
    "bryant-stories.txt": "Stories to Tell to Children",
    "burgess-busterbrown.txt": "The Adventures of Buster Bear",
    "carroll-alice.txt": "Alice's Adventures in Wonderland",
    "chesterton-ball.txt": "The Ball and the Cross",
    "chesterton-brown.txt": "The Wisdom of Father Brown",
    "chesterton-thursday.txt": "The Man Who Was Thursday",
    "edgeworth-parents.txt": "The Parent's Assistant",
    "melville-moby_dick.txt": "Moby Dick",
    "milton-paradise.txt": "Paradise Lost",
    "shakespeare-caesar.txt": "Julius Caesar",
    "shakespeare-hamlet.txt": "Hamlet",
    "shakespeare-macbeth.txt": "Macbeth",
    "whitman-leaves.txt": "Leaves of Grass",
}

# Carpeta donde NLTK dejó el corpus
candidatos = [
    os.path.join(os.environ.get("APPDATA", ""), "nltk_data", "corpora", "gutenberg"),
    os.path.expanduser("~/nltk_data/corpora/gutenberg"),
    r"C:\nltk_data\corpora\gutenberg",
]
GUT_DIR = next((p for p in candidatos if os.path.isdir(p)), None)
print("Carpeta del corpus:", GUT_DIR)

def read_doc(fileid):
    # NLTK lee estos archivos en latin-1; se usa lo mismo para obtener el mismo texto
    with open(os.path.join(GUT_DIR, fileid), encoding="latin-1") as f:
        return f.read()

fileids = sorted(f for f in os.listdir(GUT_DIR) if f.endswith(".txt"))

df_docs = pd.DataFrame({"doc_id": fileids})
df_docs["titulo"] = df_docs["doc_id"].map(titles)
df_docs["texto"] = df_docs["doc_id"].map(read_doc)
df_docs["tokens"] = df_docs["texto"].map(tokenize)

print(len(df_docs), "documentos | total tokens:", df_docs["tokens"].map(len).sum())
df_docs[["doc_id", "titulo"]].assign(n_tokens=df_docs["tokens"].map(len))

Carpeta del corpus: C:\Users\biqo\AppData\Roaming\nltk_data\corpora\gutenberg
18 documentos | total tokens: 2118910


,doc_id,titulo,n_tokens
0,austen-emma.txt,Emma,160994
1,austen-persuasion.txt,Persuasion,83615
2,austen-sense.txt,Sense and Sensibility,119912
3,bible-kjv.txt,The King James Bible,790052
4,blake-poems.txt,Poems (Blake),6837
5,bryant-stories.txt,Stories to Tell to Children,46319
6,burgess-busterbrown.txt,The Adventures of Buster Bear,15963
7,carroll-alice.txt,Alice's Adventures in Wonderland,26687
8,chesterton-ball.txt,The Ball and the Cross,82168
9,chesterton-brown.txt,The Wisdom of Father Brown,72389


In [4]:
#Celda 2: DF, IDF y estadísticas de la colección
N = len(df_docs)

# DF: cada documento cuenta una sola vez por término
df_counter = Counter()
for toks in df_docs["tokens"]:
    df_counter.update(set(toks))

# IDF de BM25
idf = {t: math.log(1 + (N - c + 0.5) / (c + 0.5)) for t, c in df_counter.items()}

# Estadísticas de la colección (se mantienen fijas para todas las consultas)
doc_len = df_docs["tokens"].map(len).to_numpy()
avgdl = doc_len.mean()

print(f"N = {N} | vocabulario = {len(df_counter)} | avgdl = {avgdl:.1f}")
for t in ["the", "whale", "ghost", "satan", "latterly"]:
    print(f"{t:<10} df={df_counter.get(t, 0):<3} idf={idf.get(t, float('nan')):.4f}")

N = 18 | vocabulario = 43494 | avgdl = 117717.2
the        df=18  idf=0.0267
whale      df=6   idf=1.0726
ghost      df=9   idf=0.6931
satan      df=7   idf=0.9295
latterly   df=1   idf=2.5390


### 2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [6]:
#Celda 1: representación dispersa y funciones de ranking
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

doc_ids = df_docs["doc_id"].tolist()
tf_docs = [Counter(t) for t in df_docs["tokens"]]

# TF-IDF + coseno (matriz dispersa)
vectorizer = TfidfVectorizer(analyzer=lambda t: t, use_idf=True,
                             smooth_idf=False, sublinear_tf=False, norm=None)
X = vectorizer.fit_transform(df_docs["tokens"])
print(type(X).__name__, X.shape, "| no nulos:", X.nnz)

K1, B = 1.2, 0.75

def rank(scores, k=10):
    """Solo scores > 0; orden por score desc y doc_id asc (como cadena)."""
    pares = [(i, s) for i, s in enumerate(scores) if s > 0]
    pares.sort(key=lambda p: (-p[1], str(doc_ids[p[0]])))
    return [(doc_ids[i], float(s)) for i, s in pares[:k]]

def tfidf_rank(query, k=10):
    toks = tokenize(query)
    if not any(t in vectorizer.vocabulary_ for t in toks):
        return []
    q = vectorizer.transform([toks])
    return rank(cosine_similarity(q, X)[0], k)

def bm25_scores(query, k1=K1, b=B):
    terms = {t for t in tokenize(query) if t in idf}
    scores = np.zeros(N)
    for i, tf in enumerate(tf_docs):
        norm = k1 * (1 - b + b * doc_len[i] / avgdl)
        for t in terms:
            f = tf.get(t, 0)
            if f:
                scores[i] += idf[t] * f * (k1 + 1) / (f + norm)
    return scores

def bm25_rank(query, k=10, k1=K1, b=B):
    if not any(t in idf for t in tokenize(query)):
        return []
    return rank(bm25_scores(query, k1, b), k)

csr_matrix (18, 43494) | no nulos: 123667


In [7]:
#Celda 2: ocho consultas, necesidad y justificación con df
queries = {
    "q1": ("whale",
           "Libros sobre ballenas (un término, df intermedio)."),
    "q2": ("the",
           "Palabra vacía muy frecuente: ve qué hace cada método con un término sin poder discriminativo."),
    "q3": ("latterly",
           "Un término muy raro (df = 1) para ver si lo domina un solo documento."),
    "q4": ("ghost murder king",
           "Tragedias con fantasmas, asesinatos y reyes."),
    "q5": ("satan heaven fall paradise",
           "Textos épicos o religiosos sobre el cielo y la caída."),
    "q6": ("marriage love family estate",
           "Novelas de costumbres sobre matrimonio, familia y herencia."),
    "q7": ("white whale captain ship sea",
           "Relatos de caza de ballenas y navegación (consulta de cinco términos)."),
    "q8": ("to be or not to be that is the question",
           "Frase célebre de Hamlet: consulta larga formada casi solo por palabras frecuentes."),
}

print(f"{'Consulta':<5}{'Términos (df)':<90}")
for k, (texto, necesidad) in queries.items():
    detalle = ", ".join(f"{t}={df_counter.get(t, 0)}" for t in dict.fromkeys(tokenize(texto)))
    print(f"{k:<5}{detalle}")
    print(f"     -> {necesidad}")

ConsultaTérminos (df)                                                                             
q1   whale=6
     -> Libros sobre ballenas (un término, df intermedio).
q2   the=18
     -> Palabra vacía muy frecuente: ve qué hace cada método con un término sin poder discriminativo.
q3   latterly=1
     -> Un término muy raro (df = 1) para ver si lo domina un solo documento.
q4   ghost=9, murder=9, king=15
     -> Tragedias con fantasmas, asesinatos y reyes.
q5   satan=7, heaven=13, fall=18, paradise=8
     -> Textos épicos o religiosos sobre el cielo y la caída.
q6   marriage=12, love=15, family=13, estate=12
     -> Novelas de costumbres sobre matrimonio, familia y herencia.
q7   white=16, whale=6, captain=9, ship=13, sea=16
     -> Relatos de caza de ballenas y navegación (consulta de cinco términos).
q8   to=18, be=18, or=18, not=18, that=18, is=18, the=18, question=17
     -> Frase célebre de Hamlet: consulta larga formada casi solo por palabras frecuentes.


In [8]:
#Celda 3: ejecutar ambos métodos y revisar tamaños
results = {}
for k, (texto, _) in queries.items():
    results[k] = {"tfidf": tfidf_rank(texto), "bm25": bm25_rank(texto)}
    r = results[k]
    print(f"{k}: TF-IDF {len(r['tfidf'])} docs | BM25 {len(r['bm25'])} docs | "
          f"1º TF-IDF: {r['tfidf'][0][0] if r['tfidf'] else '-'} | "
          f"1º BM25: {r['bm25'][0][0] if r['bm25'] else '-'}")

# Caso límite: consulta sin términos del vocabulario -> dos listas vacías
print("\nConsulta fuera del vocabulario:",
      tfidf_rank("xyzzyqwerty"), bm25_rank("xyzzyqwerty"))

q1: TF-IDF 6 docs | BM25 6 docs | 1º TF-IDF: melville-moby_dick.txt | 1º BM25: melville-moby_dick.txt
q2: TF-IDF 10 docs | BM25 10 docs | 1º TF-IDF: whitman-leaves.txt | 1º BM25: bible-kjv.txt
q3: TF-IDF 1 docs | BM25 1 docs | 1º TF-IDF: austen-emma.txt | 1º BM25: austen-emma.txt
q4: TF-IDF 10 docs | BM25 10 docs | 1º TF-IDF: shakespeare-hamlet.txt | 1º BM25: shakespeare-hamlet.txt
q5: TF-IDF 10 docs | BM25 10 docs | 1º TF-IDF: milton-paradise.txt | 1º BM25: milton-paradise.txt
q6: TF-IDF 10 docs | BM25 10 docs | 1º TF-IDF: blake-poems.txt | 1º BM25: austen-sense.txt
q7: TF-IDF 10 docs | BM25 10 docs | 1º TF-IDF: melville-moby_dick.txt | 1º BM25: melville-moby_dick.txt
q8: TF-IDF 10 docs | BM25 10 docs | 1º TF-IDF: austen-emma.txt | 1º BM25: carroll-alice.txt

Consulta fuera del vocabulario: [] []


### 3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [9]:
#Celda 1: tabla por consulta y resumen
pd.set_option("display.max_colwidth", None)

def tabla_comparacion(qk):
    t, b = results[qk]["tfidf"], results[qk]["bm25"]
    filas = []
    for pos in range(max(len(t), len(b))):
        filas.append({
            "Posición": pos + 1,
            "ID y título TF-IDF": f"{t[pos][0]} ({titles[t[pos][0]]})" if pos < len(t) else "",
            "Coseno": round(t[pos][1], 6) if pos < len(t) else "",
            "ID y título BM25": f"{b[pos][0]} ({titles[b[pos][0]]})" if pos < len(b) else "",
            "Score BM25": round(b[pos][1], 4) if pos < len(b) else "",
        })
    return pd.DataFrame(filas)

for qk, (texto, _) in queries.items():
    print(f"\n=== {qk}: '{texto}' ===")
    display(tabla_comparacion(qk))

# Resumen
resumen = []
for qk in queries:
    t_ids = [d for d, _ in results[qk]["tfidf"]]
    b_ids = [d for d, _ in results[qk]["bm25"]]
    resumen.append({
        "Consulta": qk,
        "Tamaño TF-IDF": len(t_ids),
        "Tamaño BM25": len(b_ids),
        "|T_TFIDF ∩ T_BM25|": len(set(t_ids) & set(b_ids)),
        "Mismo orden": t_ids == b_ids,
        "1º TF-IDF": t_ids[0] if t_ids else "-",
        "1º BM25": b_ids[0] if b_ids else "-",
    })
pd.DataFrame(resumen)


=== q1: 'whale' ===


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt (Moby Dick),0.108281,melville-moby_dick.txt (Moby Dick),2.3556
1,2,bryant-stories.txt (Stories to Tell to Children),0.006502,bryant-stories.txt (Stories to Tell to Children),2.2671
2,3,shakespeare-hamlet.txt (Hamlet),0.001540,whitman-leaves.txt (Leaves of Grass),1.8884
3,4,whitman-leaves.txt (Leaves of Grass),0.000757,shakespeare-hamlet.txt (Hamlet),1.8677
4,5,chesterton-ball.txt (The Ball and the Cross),0.000495,chesterton-ball.txt (The Ball and the Cross),1.6118
5,6,bible-kjv.txt (The King James Bible),0.000041,bible-kjv.txt (The King James Bible),0.5659



=== q2: 'the' ===


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,whitman-leaves.txt (Leaves of Grass),0.729582,bible-kjv.txt (The King James Bible),0.0587
1,2,melville-moby_dick.txt (Moby Dick),0.676285,whitman-leaves.txt (Leaves of Grass),0.0587
2,3,bryant-stories.txt (Stories to Tell to Children),0.668470,melville-moby_dick.txt (Moby Dick),0.0587
3,4,chesterton-brown.txt (The Wisdom of Father Brown),0.639008,chesterton-brown.txt (The Wisdom of Father Brown),0.0587
4,5,bible-kjv.txt (The King James Bible),0.619917,chesterton-ball.txt (The Ball and the Cross),0.0587
5,6,blake-poems.txt (Poems (Blake)),0.615465,bryant-stories.txt (Stories to Tell to Children),0.0587
6,7,chesterton-thursday.txt (The Man Who Was Thursday),0.594141,edgeworth-parents.txt (The Parent's Assistant),0.0587
7,8,chesterton-ball.txt (The Ball and the Cross),0.587950,chesterton-thursday.txt (The Man Who Was Thursday),0.0587
8,9,carroll-alice.txt (Alice's Adventures in Wonderland),0.561945,austen-persuasion.txt (Persuasion),0.0587
9,10,edgeworth-parents.txt (The Parent's Assistant),0.505852,austen-emma.txt (Emma),0.0587



=== q3: 'latterly' ===


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,austen-emma.txt (Emma),0.000513,austen-emma.txt (Emma),3.1639


... (7 outputs left out)

,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt (Moby Dick),0.102417,melville-moby_dick.txt (Moby Dick),5.2367
1,2,austen-persuasion.txt (Persuasion),0.033563,bryant-stories.txt (Stories to Tell to Children),4.6875
2,3,whitman-leaves.txt (Leaves of Grass),0.011832,whitman-leaves.txt (Leaves of Grass),4.6733
3,4,bryant-stories.txt (Stories to Tell to Children),0.009480,bible-kjv.txt (The King James Bible),3.3062
4,5,chesterton-brown.txt (The Wisdom of Father Brown),0.009175,shakespeare-hamlet.txt (Hamlet),3.0223
5,6,chesterton-ball.txt (The Ball and the Cross),0.007753,chesterton-ball.txt (The Ball and the Cross),2.9568
6,7,blake-poems.txt (Poems (Blake)),0.006707,chesterton-brown.txt (The Wisdom of Father Brown),2.7836
7,8,chesterton-thursday.txt (The Man Who Was Thursday),0.005773,austen-persuasion.txt (Persuasion),2.7287
8,9,carroll-alice.txt (Alice's Adventures in Wonderland),0.005542,edgeworth-parents.txt (The Parent's Assistant),2.5267
9,10,edgeworth-parents.txt (The Parent's Assistant),0.003912,chesterton-thursday.txt (The Man Who Was Thursday),2.1517



=== q8: 'to be or not to be that is the question' ===


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,austen-emma.txt (Emma),0.445579,carroll-alice.txt (Alice's Adventures in Wonderland),0.5852
1,2,austen-persuasion.txt (Persuasion),0.444161,austen-emma.txt (Emma),0.5844
2,3,edgeworth-parents.txt (The Parent's Assistant),0.441301,shakespeare-hamlet.txt (Hamlet),0.5842
3,4,austen-sense.txt (Sense and Sensibility),0.437894,chesterton-ball.txt (The Ball and the Cross),0.5837
4,5,milton-paradise.txt (Paradise Lost),0.409342,chesterton-thursday.txt (The Man Who Was Thursday),0.5837
5,6,melville-moby_dick.txt (Moby Dick),0.403852,austen-persuasion.txt (Persuasion),0.5837
6,7,shakespeare-hamlet.txt (Hamlet),0.374721,austen-sense.txt (Sense and Sensibility),0.5836
7,8,whitman-leaves.txt (Leaves of Grass),0.373524,chesterton-brown.txt (The Wisdom of Father Brown),0.5805
8,9,bryant-stories.txt (Stories to Tell to Children),0.371088,melville-moby_dick.txt (Moby Dick),0.5797
9,10,shakespeare-macbeth.txt (Macbeth),0.370984,edgeworth-parents.txt (The Parent's Assistant),0.5778


,Consulta,Tamaño TF-IDF,Tamaño BM25,|T_TFIDF ∩ T_BM25|,Mismo orden,1º TF-IDF,1º BM25
0,q1,6,6,6,False,melville-moby_dick.txt,melville-moby_dick.txt
1,q2,10,10,8,False,whitman-leaves.txt,bible-kjv.txt
2,q3,1,1,1,True,austen-emma.txt,austen-emma.txt
3,q4,10,10,7,False,shakespeare-hamlet.txt,shakespeare-hamlet.txt
4,q5,10,10,8,False,milton-paradise.txt,milton-paradise.txt
5,q6,10,10,8,False,blake-poems.txt,austen-sense.txt
6,q7,10,10,8,False,melville-moby_dick.txt,melville-moby_dick.txt
7,q8,10,10,6,False,austen-emma.txt,carroll-alice.txt


In [14]:
#Celda 2: análisis de un cambio de posición
i_b = doc_ids.index("blake-poems.txt")
i_s = doc_ids.index("austen-sense.txt")

q = vectorizer.transform([tokenize(queries["q6"][0])])
dots = (X @ q.T).toarray().ravel()
norms = np.sqrt(X.multiply(X).sum(axis=1)).A1

print(f"Producto punto: Sense/Blake = {dots[i_s]/dots[i_b]:.2f}")
print(f"Norma:          Sense/Blake = {norms[i_s]/norms[i_b]:.2f}")

def detalle(qk, d1, d2):
    texto = queries[qk][0]
    terms = list(dict.fromkeys(t for t in tokenize(texto) if t in idf))

    pos_t = {d: i + 1 for i, (d, _) in enumerate(tfidf_rank(texto, k=N))}
    pos_b = {d: i + 1 for i, (d, _) in enumerate(bm25_rank(texto, k=N))}
    sc_t = dict(tfidf_rank(texto, k=N))
    sc_b = dict(bm25_rank(texto, k=N))

    print(f"Consulta {qk}: '{texto}'  |  avgdl = {avgdl:.1f}\n")
    filas = []
    for d in (d1, d2):
        i = doc_ids.index(d)
        norm = K1 * (1 - B + B * doc_len[i] / avgdl)
        for t in terms:
            f = tf_docs[i].get(t, 0)
            aporte = idf[t] * f * (K1 + 1) / (f + norm) if f else 0.0
            filas.append({"Documento": d, "|d|": doc_len[i], "Término": t, "f(t,d)": f,
                          "df(t)": df_counter[t], "IDF": round(idf[t], 4),
                          "Aporte BM25": round(aporte, 4)})
    display(pd.DataFrame(filas))

    for d in (d1, d2):
        print(f"{d}: |d|/avgdl = {doc_len[doc_ids.index(d)]/avgdl:.3f} | "
              f"TF-IDF: pos {pos_t.get(d, '-')}, coseno {sc_t.get(d, 0):.6f} | "
              f"BM25: pos {pos_b.get(d, '-')}, score {sc_b.get(d, 0):.4f}")

detalle("q6", "blake-poems.txt", "austen-sense.txt")

Producto punto: Sense/Blake = 8.89
Norma:          Sense/Blake = 15.96
Consulta q6: 'marriage love family estate'  |  avgdl = 117717.2



,Documento,|d|,Término,"f(t,d)",df(t),IDF,Aporte BM25
0,blake-poems.txt,6837,marriage,1,12,0.4187,0.6812
1,blake-poems.txt,6837,love,29,15,0.2036,0.4425
2,blake-poems.txt,6837,family,0,13,0.3417,0.0000
3,blake-poems.txt,6837,estate,0,12,0.4187,0.0000
4,austen-sense.txt,119912,marriage,44,12,0.4187,0.8964
5,austen-sense.txt,119912,love,77,15,0.2036,0.4409
6,austen-sense.txt,119912,family,83,13,0.3417,0.7410
7,austen-sense.txt,119912,estate,19,12,0.4187,0.8657


blake-poems.txt: |d|/avgdl = 0.058 | TF-IDF: pos 1, coseno 0.022360 | BM25: pos 14, score 1.1237
austen-sense.txt: |d|/avgdl = 1.019 | TF-IDF: pos 2, coseno 0.012456 | BM25: pos 1, score 2.9440


**Resumen de la comparación.** Los dos métodos coinciden en el primer documento en 5 de las 8 consultas (difieren en q2, q6 y q8). La intersección de los top-10 va de 6 a 8 documentos (q1: 6 de 6; q3: 1 de 1), y en ninguna consulta con más de un resultado el orden completo es idéntico.

**Cambio de posición (q6: "marriage love family estate").** *Poems* (Blake) es 1º con TF-IDF + coseno (0,0224) y 14º con BM25 (1,1237); *Sense and Sensibility* es 2º con coseno (0,0125) y 1º con BM25 (2,9440).
- BM25: Blake tiene |d| = 6.837 (|d|/avgdl = 0,058), `marriage` con f = 1 (aporte 0,68), `love` con f = 29 (aporte 0,44, saturado, IDF = 0,2036) y f = 0 en `family` y `estate`. Sense tiene |d| = 119.912 (|d|/avgdl = 1,019) y aportes de 0,90, 0,44, 0,74 y 0,87 en los cuatro términos. BM25 premia cubrir varios términos y satura las repeticiones, por lo que Sense gana.
- TF-IDF + coseno: el producto punto de Sense es ≈ 8,9 veces mayor que el de Blake, pero su norma es ≈ 16,0 veces mayor. Como TF no está normalizado, el coseno penaliza a los documentos largos y favorece a Blake.

**Efecto de la saturación y la longitud.** En q1, *Moby Dick* y *Stories* quedan muy separados con coseno (0,108 vs 0,0065) pero casi empatados con BM25 (2,36 vs 2,27). En q2 y q8, los términos de IDF casi nulo hacen que BM25 dé scores prácticamente iguales, y el orden depende solo de diferencias mínimas de longitud.

### 4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [11]:
#Celda 1: posiciones y scores en las tres configuraciones
QK = "q6"
texto = queries[QK][0]

configs = {
    "Base (k1=1.2, b=0.75)": (1.2, 0.75),
    "Sin longitud (k1=1.2, b=0)": (1.2, 0.0),
    "Más repeticiones (k1=2.0, b=0.75)": (2.0, 0.75),
}

# Ranking completo (todos los docs con score > 0) por configuración
pos, sc = {}, {}
for nombre, (k1, b) in configs.items():
    r = bm25_rank(texto, k=N, k1=k1, b=b)
    pos[nombre] = {d: i + 1 for i, (d, _) in enumerate(r)}
    sc[nombre] = dict(r)

terms = list(dict.fromkeys(t for t in tokenize(texto) if t in idf))

filas = []
for d in doc_ids:
    i = doc_ids.index(d)
    fila = {"Documento": d, "|d|/avgdl": round(doc_len[i] / avgdl, 3)}
    for t in terms:
        fila[f"f({t})"] = tf_docs[i].get(t, 0)
    for nombre in configs:
        fila[f"Pos {nombre.split(' (')[0]}"] = pos[nombre].get(d, "-")
    filas.append(fila)

tabla_pos = pd.DataFrame(filas).sort_values("Pos Base")
display(tabla_pos)

# Cambios de posición respecto a la base
base = pos["Base (k1=1.2, b=0.75)"]
for nombre in list(configs)[1:]:
    movidos = [(d, base[d], pos[nombre][d]) for d in base if d in pos[nombre] and base[d] != pos[nombre][d]]
    print(f"\n{nombre}: {len(movidos)} documentos cambian de posición")
    for d, p0, p1 in sorted(movidos, key=lambda x: x[1]):
        print(f"  {d:<28} {p0:>2} -> {p1:>2}")

,Documento,|d|/avgdl,f(marriage),f(love),f(family),f(estate),Pos Base,Pos Sin longitud,Pos Más repeticiones
2,austen-sense.txt,1.019,44,77,83,19,1,1,1
1,austen-persuasion.txt,0.710,27,42,80,6,2,3,2
0,austen-emma.txt,1.368,35,117,77,4,3,4,3
3,bible-kjv.txt,6.711,19,310,123,17,4,2,4
17,whitman-leaves.txt,1.053,8,272,5,2,5,5,5
13,milton-paradise.txt,0.680,3,99,1,1,6,9,8
9,chesterton-brown.txt,0.615,0,9,45,5,7,8,6
11,edgeworth-parents.txt,1.425,0,83,39,5,8,7,7
10,chesterton-thursday.txt,0.495,4,8,5,0,9,10,9
12,melville-moby_dick.txt,1.835,2,24,11,1,10,6,10



Sin longitud (k1=1.2, b=0): 10 documentos cambian de posición
  austen-persuasion.txt         2 ->  3
  austen-emma.txt               3 ->  4
  bible-kjv.txt                 4 ->  2
  milton-paradise.txt           6 ->  9
  chesterton-brown.txt          7 ->  8
  edgeworth-parents.txt         8 ->  7
  chesterton-thursday.txt       9 -> 10
  melville-moby_dick.txt       10 ->  6
  shakespeare-hamlet.txt       12 -> 13
  chesterton-ball.txt          13 -> 12

Más repeticiones (k1=2.0, b=0.75): 5 documentos cambian de posición
  milton-paradise.txt           6 ->  8
  chesterton-brown.txt          7 ->  6
  edgeworth-parents.txt         8 ->  7
  bryant-stories.txt           11 -> 12
  shakespeare-hamlet.txt       12 -> 11


In [12]:
#Celda 2: aportes por término para documentos de longitudes distintas
def aportes(d, k1, b):
    i = doc_ids.index(d)
    norm = k1 * (1 - b + b * doc_len[i] / avgdl)
    out = {}
    for t in terms:
        f = tf_docs[i].get(t, 0)
        out[t] = idf[t] * f * (k1 + 1) / (f + norm) if f else 0.0
    return out

docs_estudio = ["blake-poems.txt", "austen-sense.txt", "bible-kjv.txt"]

filas = []
for d in docs_estudio:
    for nombre, (k1, b) in configs.items():
        a = aportes(d, k1, b)
        fila = {"Documento": d, "Configuración": nombre.split(" (")[0]}
        fila.update({t: round(v, 4) for t, v in a.items()})
        fila["Total"] = round(sum(a.values()), 4)
        filas.append(fila)

display(pd.DataFrame(filas))

,Documento,Configuración,marriage,love,family,estate,Total
0,blake-poems.txt,Base,0.6812,0.4425,0.0000,0.0000,1.1237
1,blake-poems.txt,Sin longitud,0.4187,0.4301,0.0000,0.0000,0.8488
2,blake-poems.txt,Más repeticiones,0.7915,0.5987,0.0000,0.0000,1.3901
3,austen-sense.txt,Base,0.8964,0.4409,0.7410,0.8657,2.9440
4,austen-sense.txt,Sin longitud,0.8967,0.4410,0.7411,0.8664,2.9453
5,austen-sense.txt,Más repeticiones,1.2008,0.5951,1.0008,1.1350,3.9317
6,bible-kjv.txt,Base,0.6907,0.4389,0.7150,0.6709,2.5155
7,bible-kjv.txt,Sin longitud,0.8664,0.4462,0.7446,0.8604,2.9176
8,bible-kjv.txt,Más repeticiones,0.8072,0.5907,0.9441,0.7746,3.1166


In [15]:
#Caso de Blake
for nombre, (k1, b) in configs.items():
    r = bm25_rank(texto, k=N, k1=k1, b=b)
    print(nombre)
    for p in (12, 13, 14):       # posiciones 13, 14 y 15
        print(f"  {p+1:>2}. {r[p][0]:<26} {r[p][1]:.4f}")

Base (k1=1.2, b=0.75)
  13. chesterton-ball.txt        1.5437
  14. blake-poems.txt            1.1237
  15. carroll-alice.txt          0.8834
Sin longitud (k1=1.2, b=0)
  13. shakespeare-hamlet.txt     1.4256
  14. blake-poems.txt            0.8488
  15. carroll-alice.txt          0.6617
Más repeticiones (k1=2.0, b=0.75)
  13. chesterton-ball.txt        1.8514
  14. blake-poems.txt            1.3901
  15. carroll-alice.txt          1.0344


**Efecto de los parámetros (q6: "marriage love family estate").**

| Ejecución | Docs que cambian de posición | Efecto principal |
|---|---:|---|
| Base (k₁ = 1,2; b = 0,75) | – | Referencia |
| Sin longitud (b = 0) | 10 | La Biblia sube de 4º a 2º; Moby Dick de 10º a 6º; Paradise Lost baja de 6º a 9º |
| Más repeticiones (k₁ = 2,0) | 5 | Cambios solo entre las posiciones 6 y 12 |

- **b controla la normalización por longitud.** Con b = 0, el aporte de `marriage` en la Biblia (|d|/avgdl = 6,71) sube de 0,6907 a 0,8664, y el de Blake (|d|/avgdl = 0,058) baja de 0,6812 a 0,4187. *Sense and Sensibility* (|d|/avgdl = 1,019) casi no cambia (2,9440 → 2,9453), porque cuando |d| ≈ avgdl el término de longitud vale casi 1.
- **k₁ controla la saturación.** Con k₁ = 2,0 el máximo de cada término sube (de 2,2·IDF a 3,0·IDF) y los términos con f alta ganan más: `family` en *Sense* (f = 83) sube un 35 %, mientras que `marriage` en Blake (f = 1) sube un 16 %. En `love`, tanto Blake (f = 29) como *Sense* (f = 77) están cerca del techo (0,5987 y 0,5951), por lo que la diferencia de repeticiones sigue sin pesar.
- **Documentos que no se mueven.** *Sense* es 1º en las tres configuraciones: su score cambia (2,9440; 2,9453; 3,9317), pero acumula aportes altos en los cuatro términos y mantiene el liderazgo. Blake es 14º en las tres: su score cambia (1,1237; 0,8488; 1,3901), pero también cambian los de sus vecinos, así que el orden se conserva. Con b = 0, Blake baja a 0,8488 y *Alice* (15ª) a 0,6617, una diferencia de 0,187 que se mantiene positiva; con k₁ = 2,0, ambos suben (1,3901 y 1,0344) y la diferencia es de 0,356. Blake queda por encima de *Alice* porque acumula dos términos de la consulta (`marriage` con f = 1 y `love` con f = 29), mientras que *Alice* solo coincide con aportes pequeños en `love` (f = 3) y `family` (f = 1).